## Concept and motivation

**Ball Carrier Open Space (BCOS)** is the space a player in possession can navigate without interference from an opponent. [Bandara et al. (2025)](https://doi.org/10.1007/s42979-025-03815-7) define it as the unoccupied area surrounding the ball carrier and use it to study how open space varies across the pitch and over a match.

PRESS formalizes this idea in a **competing-risks framework analogous to pitch control**. Given an observed tracking state, we consider a set of possible destinations for the carrier. Each destination defines a hypothetical straight-line route: the carrier attempts to reach its endpoint, while opposing players may intercept him at any position along the route. Arrival at the destination and interception events compete through a common survival process, and the first event determines the outcome.

Movement constraints and uncertainty in player arrival times set the relative likelihood of these events. The model assigns each destination $z$ an **accessibility probability** $p_B(z)$, the probability that the carrier reaches $z$ before any interception. Evaluating $p_B(z)$ across the pitch gives an accessibility surface, which shows where space is available and how extensive it is.

The spatial integral of the surface is the **probability-weighted accessible area**,

$$A_{\mathrm{BCOS}}(s_0)=\int_{\mathcal D_{40}}p_B(z)\,dz\qquad[\mathrm{m}^2],$$

where $s_0$ is the observed tracking state and $\mathcal D_{40}$ is the pitch-clipped region within 40 m of the carrier. The area condenses the surface into a single state-level quantity, while the surface keeps the spatial organization of opportunities. Both are conditional on the observed tracking state and do not require specifying the carrier's subsequent action.

BCOS is implemented as the optional `PRESS.extras.BCOS` component. Its scalar area is used as a predictor in the [action-selection model](../../04_action_selection/04_action_selection.ipynb), complementing the local loss probabilities of the [ball-drive risk model](../../01_ball_drive_risk/01_ball_drive_risk.ipynb).

## Mathematical formulation

The construction has four steps: the arrival-time distribution of each player at each point, the carrier's possible positions along a route, the competing interception hazards that give the accessibility probability, and the integration over destinations.

### Arrival-time distributions

Let $r_B$ denote the carrier's position. For a candidate destination $z\in\mathcal D_{40}$, the straight-line route is

$$x(\ell;z)=r_B+\frac{\ell}{L_z}(z-r_B),\qquad 0\leq\ell\leq L_z,\qquad L_z=\|z-r_B\|.$$

The movement model assigns each player $j$ an effective arrival time $\tau_j(x)$ at a location $x$, which accounts for the player's initial position and velocity, acceleration-limited movement and a turning delay. The turning component is $t_{\mathrm{turn},j}=u_j\theta_j/a_{\mathrm{turn}}$, where $u_j$ is the speed of player $j$, $\theta_j$ the angle through which the player must turn to head towards $x$ and $a_{\mathrm{turn}}$ the turning parameter, so that $\tau_j(x)=t_{\mathrm{turn},j}(x)+t_{\mathrm{move},j}(x)$. These movement assumptions follow the kinematic model used elsewhere in PRESS.

Following the probabilistic arrival-time formulation of pitch control ([Spearman et al., 2017](https://static.hudl.com/craft/downloads/SSAC17-Physics-Based-Modeling-of-Pass-Probabilities-in-Soccer.pdf); [@spearman2018beyond]), uncertainty around the effective arrival time is represented by a logistic cumulative distribution,

$$F_j(x,t)=\sigma\!\left(\alpha[t-\tau_j(x)]\right),\qquad
\alpha=\frac{\pi}{\sqrt{3}\,s},$$

where $\sigma$ is the logistic function and $s$ is the standard deviation of arrival-time uncertainty. The associated arrival intensity,

$$\lambda_j(t\mid x)=\frac{\partial_t F_j(x,t)}{1-F_j(x,t)}
=\alpha F_j(x,t),$$

is the instantaneous rate at which player $j$ arrives at $x$, given that the player has not arrived yet.

### Carrier occupancy along a candidate route

An interception can occur before the carrier reaches the destination, so the model needs the carrier's possible positions along the whole route. For a fixed destination $z$, the carrier's arrival-time distribution induces a spatial distribution over route positions: the probability of having reached the point at distance $\ell$ by time $t$ is $F_B(x(\ell;z),t)$. Since the effective arrival time increases with distance along the route, $F_B$ decreases in $\ell$, and its spatial gradient gives the carrier's pre-arrival occupancy density,

$$w_B(\ell,t;z)=-\frac{\partial}{\partial\ell}F_B(x(\ell;z),t),\qquad 0<\ell<L_z.$$

The carrier starts at $r_B$, so the spatial measure also includes the probability mass still at the origin,

$$\mu_{B,t,z}(d\ell)=
\big[1-F_B(r_B,t)\big]\delta_0(d\ell)
+w_B(\ell,t;z)\,d\ell,\qquad 0\leq\ell<L_z.$$

For monotone arrival-time profiles, the total mass is $1-F_B(z,t)$, the probability that the carrier has not yet reached the destination. The measure describes where the carrier may be along the route at each time and supplies the spatial weights for the interception events.

### Interception hazards and competing events

For defender $i$, the arrival intensity $\lambda_i(t\mid x(\ell;z))$ measures the defender's ability to reach a possible carrier location at time $t$. Weighting it by the carrier's occupancy measure gives the route-level interception hazard,

$$h_i(t;z)=\int_{[0,L_z)}
\lambda_i\!\left(t\mid x(\ell;z)\right)\,
\mu_{B,t,z}(d\ell).$$

Each defender therefore contributes through all possible interception locations, each weighted by the probability that the carrier is there. In a spatial discretization, the defender–location contributions are

$$h_{ik}(t;z)=w_k(t;z)\,\lambda_i(t\mid x_k),\qquad
h_i(t;z)=\sum_k h_{ik}(t;z),$$

where $w_k$ is the probability mass of the corresponding route interval.

The carrier's arrival at the destination is a further competing event, with hazard

$$h_B(t;z)=\lambda_B(t\mid z)=\alpha F_B(z,t).$$

The competing events are thus successful arrival and all defender–location interceptions. Their hazards add,

$$H(t;z)=h_B(t;z)+\sum_i h_i(t;z),$$

and define the common survival function

$$U_z(t)=\exp\!\left[-\int_0^t H(u;z)\,du\right],$$

the probability that neither the arrival nor an interception has resolved the route by time $t$. The probability of accessing destination $z$ is the cumulative incidence of successful arrival,

$$\boxed{p_B(z)=\int_0^T U_z(t)\,h_B(t;z)\,dt},$$

where $T$ is the time horizon of the calculation.

The carrier therefore succeeds only when arrival precedes every interception. An interception by any defender resolves the route as unsuccessful, and neither the defender's identity nor the interception location enters the binary outcome. The construction is conditional on the observed tracking state: arrival-time uncertainty and carrier occupancy are specified from that state, and interception contributions are combined additively. The model does not simulate defensive reactions to the hypothetical route or coordinated future trajectories.

### Accessibility surface and accessible area

Repeating the calculation for all candidate destinations gives the field $z\mapsto p_B(z)$, whose integral is the BCOS area,

$$\boxed{A_{\mathrm{BCOS}}(s_0)=\int_{\mathcal D_{40}}p_B(z)\,dz}.$$

The model thus involves two spatial integrations. The first runs along each route and yields the competing interception hazards, which capture the exposure to defenders along a potential movement. The second runs across candidate destinations and measures the extent of the accessible region. Since $0\leq p_B(z)\leq1$, the area is the area of the target domain weighted by the probability of reaching each location.

## Example: accessibility on a real tracking frame

We use SkillCorner Open Data from **Brisbane FC–Perth Glory, 21 December 2024** (match 1925299) ([SkillCorner Open Data](https://github.com/SkillCorner/opendata)). Frame 15032 (25:02.20) is the first eligible production checkpoint at or after minute 25, selected without conditioning on its BCOS value or outcome. The carrier is `home_17`, near the attacking boundary.

For this state the accessibility surface covers **1,721 candidate destinations**, and the probability-weighted accessible area is approximately **46.45 m²**. The figure shows how the probability of reaching each location varies with the carrier's position and the opponents' movement constraints. The code recomputes the field from processed tracking when available, and otherwise shows the previously exported figure.

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PRESS import PRESS
from PRESS.extras import BCOS
from PRESS.visualization import plot_bcos

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").is_file())
summary = json.loads((root / "data/examples/bcos/example_summary.json").read_text())
default_dir = root.parent / "opendata/data/processed/2026-10-15_training_v5_outfield/matches/1925299"
match_dir = Path(os.environ.get("PRESS_BCOS_MATCH_DIR", str(default_dir)))
frame_id = summary["frame_id"]
if (match_dir / "1925299_tracking.parquet").is_file():
    press = PRESS.from_skillcorner(match_dir)
    bcos = BCOS(press.tracking_data)
    result = bcos.compute_bcos(frame_id)  # full40m/1m/0.03125m/0.04s defaults
    print("Actual carrier:", press.get_state(frame_id)["player"] if "player" in press.get_state(frame_id) else summary["carrier_id"])
    print("BCOS area (m²):", result["BCOS area"])
    print("Targets:", len(result["grid"]))
    print("Maximum probability-mass error:", result["max_mass_error"])
    print("Difference from the verified real-case export (m²):", result["BCOS area"] - summary["bcos_area_m2"])
    assert np.isclose(result["BCOS area"], summary["bcos_area_m2"], rtol=0, atol=1e-6)
    plot_bcos(bcos, frame_id, figsize=(10, 6), title_fontsize=11, legend_fontsize=9)
    fig = plt.gcf()
    fig.axes[0].set_title("Brisbane FC–Perth Glory · real tracking frame 15032\nBCOS area = %.2f m²" % result["BCOS area"])
else:
    print("Real-case local data unavailable: showing the verified rendered calculation.")
    print("BCOS area (m²):", summary["bcos_area_m2"])
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(plt.imread(root / "figures/vignette_10_bcos.png"))
    ax.axis("off")
    plt.tight_layout()


## Spatiotemporal statistics for one match

To show how accessible space varies with carrier location and match time, we summarize **1,928 production action-choice checkpoints** from **931 individual possessions** in the same match (Brisbane: 1,238 checkpoints from 592 possessions; Perth: 690 from 339). The sample comprises outfield, no-shot action-choice checkpoints from the observed tracking sequence. Brisbane's mean and median BCOS are **311.42 and 101.99 m²**, and Perth's are **226.64 and 73.64 m²**; the gap between means and medians reflects a skewed distribution with some very open states.

### Spatial distribution by carrier position

Both teams are oriented to attack right, including after halftime. Carrier origins are grouped into an **8 × 5 display grid**, separate from the 1 m destination grid of the model. For team $a$ and origin cell $g$,

$$\bar A_{a,g}=\frac{1}{n_{a,g}}\sum_{j:\,a_j=a,\,r_{B,j}\in g}A_{\mathrm{BCOS}}(s_j).$$

Colors show the **mean BCOS at checkpoints whose carrier occupies that cell**, and they do not show the probability of accessing the cell or the time spent there. Labels count distinct individual possessions, and both teams share one color scale. Cells with fewer than five distinct possessions are hatched, and empty cells are blank. The 8 × 5 resolution is a compromise, since finer bins have weaker support. Seven origin states slightly outside the pitch are excluded from the display grid and kept in the model and temporal summaries.

![Mean BCOS by carrier origin, with possession counts and a common color scale.](../../../figures/bcos_match_spatial.png)

### Temporal variation by pitch third

Following the interest of [Bandara et al. (2025)](https://doi.org/10.1007/s42979-025-03815-7) in defensive, central and attacking thirds, each half is divided into five time windows. For team $a$, carrier-origin third $q$ and window $b$,

$$\bar A_{a,q,b}=\frac{1}{n_{a,q,b}}\sum_{j:\,a_j=a,\,r_{B,j}\in q,\,t_j\in b}A_{\mathrm{BCOS}}(s_j).$$

Each curve reports the mean **full-domain** BCOS *conditional on the carrier being in that third*, and the target surface is not partitioned into thirds. Windows with fewer than five individual possessions are left blank, without interpolation, and the two halves are displayed separately.

![Mean BCOS by team, carrier third and match-time window.](../../../figures/bcos_match_temporal.png)

| Carrier-origin third | Brisbane mean BCOS (m²) | Perth mean BCOS (m²) |
|---|---:|---:|
| Defensive | 548.31 | 277.33 |
| Central | 365.72 | 260.13 |
| Attacking | 81.64 | 68.23 |

Both teams have substantially less accessible area at attacking-third checkpoints than at defensive-third checkpoints, which is consistent with a combination of opponent geometry and the pitch boundary that the aggregate alone cannot separate. Brisbane has considerably more accessible space in the sampled defensive and central states, and the difference near the opposing goal is smaller. Perth's attacking-third curve is incomplete in the final window, where the sample has 6 checkpoints from 3 possessions against Brisbane's 116 from 45. The case study profiles the location and evolution of available space, and says nothing about chance quality, realized actions or tactical causation. [Phase summaries](../../../data/examples/bcos/match_phase_summary.csv).

### Reproducing the descriptive figures

The cell below rebuilds the figures from the repository's public aggregate tables. Recomputing BCOS itself requires processed tracking data. See the [vignette provenance](../../../docs/bcos-vignette-provenance.json) and the [aggregate definitions](../../../data/examples/bcos/README.md).

In [ ]:
import pandas as pd
from scripts.build_bcos_vignette_figures import main as rebuild_bcos_figures
rebuild_bcos_figures()
print(pd.read_csv(root / "data/examples/bcos/team_summary.csv").to_string(index=False))
print("Spatial and temporal cell definitions:")
print(json.loads((root / "data/examples/bcos/metadata.json").read_text())["sampling"])


## Scope and references

BCOS assumes straight hypothetical routes, effective movement limits, logistic arrival uncertainty and an additive competing-hazard approximation across defenders and route locations, with arrival uncertainty and carrier position modeled separately. Body posture, ball-control skill, adaptive defensive reactions and realized route choice are outside the model, and the match-level summaries refer to the sampled action-choice checkpoints. The scalar area may be complemented by geometric summaries such as forward-accessible share, angular concentration and radial extent.

Concept and regional temporal analysis: [Bandara et al. (2025)](https://doi.org/10.1007/s42979-025-03815-7). Arrival/control framework: Spearman ([Spearman et al., 2017](https://static.hudl.com/craft/downloads/SSAC17-Physics-Based-Modeling-of-Pass-Probabilities-in-Soccer.pdf); [@spearman2018beyond]). Turning and direction-sensitive movement motivation: Pleuler and Bekkers [@pleuler2020handbook; @bekkers2025]. Movement-scale context: Bradley, Carling and Benhassen et al. [@bradley2009running; @carling2010ballrunning; @benhassen2026dribbling]. Tracking: [SkillCorner Open Data](https://github.com/SkillCorner/opendata). The spatial interception construction and its numerical implementation are PRESS-specific derivations.

[Spatial support](../../../data/examples/bcos/grid_support.csv) · [spatial statistics](../../../data/examples/bcos/spatial_cells.csv) · [temporal statistics](../../../data/examples/bcos/temporal_thirds.csv).